In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
from sklearn.metrics import log_loss
from dixon_coles import fit_dc, walk_forward

df = pd.read_csv("../data/processed/laliga_matches.csv",
                 parse_dates=["Date"], dtype={"Season": str})
df["y"] = df["FTR"].map({"H": 0, "D": 1, "A": 2})

ini = pd.Timestamp("2023-08-01")
m = fit_dc(df[df["Date"] < ini], ini, xi=0.002, ridge=1.0)

fuerza = pd.DataFrame({"ataque": m["att"], "defensa": m["def"]},
                      index=sorted(m["idx"], key=m["idx"].get))
print(fuerza.sort_values("ataque", ascending=False).round(2).head(6))
print("ventaja local:", round(m["home"], 3), " rho:", round(m["rho"], 3))

             ataque  defensa
Real Madrid    0.48     0.35
Barcelona      0.45     0.48
Ath Madrid     0.39     0.34
Villarreal     0.31     0.15
Girona         0.25    -0.09
Levante        0.14    -0.30
ventaja local: 0.297  rho: 0.009


In [2]:
VAL = ["2122", "2223"]
for xi in [0.0, 0.001, 0.002, 0.004]:
    sub = walk_forward(df, VAL, xi=xi, ridge=1.0)
    print(xi, round(log_loss(sub["y"], sub[["pH", "pD", "pA"]].values, labels=[0, 1, 2]), 4))

0.0 1.0063
0.001 1.0016
0.002 1.0023
0.004 1.0081


In [3]:
TEST = ["2324", "2425", "2526"]
sub_te = walk_forward(df, TEST, xi=0.001, ridge=1.0)
P_dc = sub_te[["pH", "pD", "pA"]].values
print("Dixon-Coles test:", round(log_loss(sub_te["y"], P_dc, labels=[0, 1, 2]), 4),
      "| partidos:", len(sub_te))

Dixon-Coles test: 0.9722 | partidos: 1140


In [4]:
from elo import add_elo
from sklearn.linear_model import LogisticRegression

TRAIN = ["1617","1718","1819","1920","2021","2122","2223"]

# Elo ajustado (regress=0), logística entrenada solo con train
d_elo = add_elo(df, k=20, home_adv=60, regress=0.0)
tr = d_elo[d_elo["Season"].isin(TRAIN)]
te = d_elo[d_elo["Season"].isin(TEST)].copy()
lr = LogisticRegression(max_iter=1000).fit(tr[["elo_diff"]], tr["y"])
te[["eH", "eD", "eA"]] = lr.predict_proba(te[["elo_diff"]])

# Unir con Dixon-Coles por clave (así no dependemos del orden de las filas)
keys = ["Date", "HomeTeam", "AwayTeam"]
t = te.merge(sub_te[keys + ["pH", "pD", "pA"]], on=keys, how="inner")
print("Partidos tras unir:", len(t))   # debe ser 1140

# Mercado normalizado
inv = 1 / t[["AvgCH", "AvgCD", "AvgCA"]].values
P_mkt = inv / inv.sum(axis=1, keepdims=True)

P_elo = t[["eH", "eD", "eA"]].values
P_dc  = t[["pH", "pD", "pA"]].values
P_ens = (P_elo + P_dc) / 2          # ensamble 50/50, sin parámetros que ajustar
y = t["y"].values

Partidos tras unir: 1140


## Log-loss y bootstrap pareado por bloques semanales

In [6]:
def perdida(P):
    return -np.log(P[np.arange(len(y)), y])

L = {"elo": perdida(P_elo), "dc": perdida(P_dc),
     "ens": perdida(P_ens), "mkt": perdida(P_mkt)}
for k_, v in L.items():
    print(f"{k_:4s} log-loss = {v.mean():.4f}")

# Bloques semanales
bloque = t["Date"].dt.to_period("W").astype(str).values
g = pd.DataFrame({"b": bloque, "n": 1, **L}).groupby("b").sum()
rng = np.random.default_rng(42)
B, nb = 10000, len(g)

def boot(a, b):
    out = []
    for _ in range(B):
        tot = g.iloc[rng.integers(0, nb, nb)].sum()
        out.append((tot[a] - tot[b]) / tot["n"])
    lo, hi = np.percentile(out, [2.5, 97.5])
    print(f"{a} - {b}: media={np.mean(out):+.4f}  IC95%=[{lo:+.4f}, {hi:+.4f}]")

boot("dc", "elo")
boot("ens", "elo")
boot("ens", "dc")
boot("dc", "mkt")
boot("ens", "mkt")

elo  log-loss = 0.9732
dc   log-loss = 0.9722
ens  log-loss = 0.9696
mkt  log-loss = 0.9537
dc - elo: media=-0.0010  IC95%=[-0.0098, +0.0078]
ens - elo: media=-0.0036  IC95%=[-0.0083, +0.0008]
ens - dc: media=-0.0026  IC95%=[-0.0067, +0.0016]
dc - mkt: media=+0.0184  IC95%=[+0.0084, +0.0282]
ens - mkt: media=+0.0159  IC95%=[+0.0065, +0.0254]


## Dixon-Coles y ensamble

Modelo Dixon-Coles (ataque/defensa por equipo, corrección de marcadores bajos,
pesos temporales con ξ = 0,001 elegido en validación, reajuste mensual con
solo partidos anteriores). El ensamble promedia al 50 % las probabilidades de
Elo y Dixon-Coles, sin parámetros ajustados con el test.

| Modelo | Log-loss (test 2023/24-2025/26) |
|---|---|
| Elo ajustado | 0,9732 |
| Dixon-Coles | 0,9722 |
| Ensamble 50/50 | 0,9696 |
| Mercado (AvgC) | 0,9537 |

Bootstrap pareado por bloques semanales (10.000 remuestras):

| Comparación | Diferencia | IC 95 % |
|---|---|---|
| DC − Elo | −0,0010 | [−0,0098, +0,0078] |
| Ensamble − Elo | −0,0036 | [−0,0083, +0,0008] |
| Ensamble − DC | −0,0026 | [−0,0067, +0,0016] |
| DC − mercado | +0,0184 | [+0,0084, +0,0282] |
| Ensamble − mercado | +0,0159 | [+0,0065, +0,0254] |

**Lectura:** los modelos propios no se distinguen entre sí; el ensamble es el
mejor en cifra puntual. El mercado sigue siendo mejor de forma distinguible
(brecha ≈ 0,016). Con solo resultados históricos parece haber un techo en
torno a 0,97; la información restante (alineaciones, lesiones) no está en
nuestros datos.

**Limitaciones:** cinco comparaciones sobre los mismos datos (riesgo de falsos
positivos), un único corte train/test, `ridge` fijado a mano.